In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

from torch.utils.data import DataLoader , Dataset
import torch
import torch.nn as nn
import torch.optim as optim

import matplotlib.pyplot as plt

In [4]:
torch.manual_seed(42)

In [5]:
device = torch.device('cuda')
print(f"Using device : {device}")

Using device : cuda


In [6]:
df = pd.read_csv('/content/fashion-mnist_train.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
x = df.iloc[: , 1:].values

y = df.iloc[: , 0].values

In [8]:
X_train , X_test , y_train , y_test = train_test_split(x , y , test_size = 0.2 , random_state = 42)

In [9]:
# Transformations
from torchvision.transforms import transforms

custom_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485 , 0.456 , 0.406] , std = [0.229 , 0.224 , 0.225])
])


In [10]:
# Python Image Library
from PIL import Image
import numpy as np

class customDataset(Dataset):

  def __init__(self , features , labels , transform):
    self.features = features
    self.labels = labels
    self.transform = transform

  def __len__(self):
    return len(self.features)

  def __getitem__(self , index):
    # reshape to (28 , 28)
    image = self.features[index].reshape(28 , 28)

    # change datatype to np.uint8
    image = image.astype(np.uint8)

    # change 1-D to 3-D
    image = np.stack([image]*3 , axis = -1)

    # change the image into the PIL image
    image = Image.fromarray(image)

    #apply transforms
    image = self.transform(image)

    # return
    return image , torch.tensor(self.labels[index] , dtype = torch.long)


In [11]:
train_dataset = customDataset(X_train , y_train , transform = custom_transform)
test_dataset = customDataset(X_test , y_test , transform = custom_transform)

In [12]:
train_loader = DataLoader(train_dataset , batch_size = 128 , shuffle = True , pin_memory = True)
test_loader = DataLoader(test_dataset , batch_size = 128 , shuffle = False , pin_memory = True)

In [13]:
# Fetch pretrained model

import torchvision.models as models

vgg16 = models.vgg16(pretrained = True)

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:07<00:00, 78.0MB/s]


In [14]:
vgg16

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [15]:
for param in vgg16.features.parameters():
  param.requires_grad = False

In [16]:
vgg16.classifier = nn.Sequential(
        nn.Linear(25088, 4096),
        nn.BatchNorm1d(4096),
        nn.ReLU(),
        nn.Dropout(p = 0.4),

        nn.Linear(4096 , 1028),
        nn.BatchNorm1d(1028),
        nn.ReLU(),
        nn.Dropout(p = 0.4),

        nn.Linear(1028 , 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(p = 0.4),

        nn.Linear(512 , 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(p = 0.2),

        nn.Linear(256 , 10)
    )

In [17]:
vgg16

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [18]:
vgg16 = vgg16.to(device)

In [19]:
epochs = 7
learning_rate = 0.0001

In [20]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(vgg16.classifier.parameters() , lr = learning_rate)

In [21]:
for epoch in range(epochs):
  total_epoch_loss = 0
  for batch_features , batch_labels in train_loader:

    batch_features , batch_labels = batch_features.to(device) , batch_labels.to(device)
    # forward pass
    outputs = vgg16(batch_features)

    # loss function
    loss = criterion(outputs , batch_labels)

    # back-pass
    optimizer.zero_grad()
    loss.backward()

    # Weights updation
    optimizer.step()

    total_epoch_loss += loss.item()

  avg_loss = total_epoch_loss / len(train_loader)
  print(f"Epoch : {epoch + 1} , Loss : {avg_loss}")

Epoch : 1 , Loss : 0.4659136606057485
Epoch : 2 , Loss : 0.20657461112737655
Epoch : 3 , Loss : 0.13761512644092241
Epoch : 4 , Loss : 0.09371056983868281
Epoch : 5 , Loss : 0.07122146904468536
Epoch : 6 , Loss : 0.05226677988221248
Epoch : 7 , Loss : 0.04666494894151886


In [22]:
vgg16.eval()

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [23]:
total = 0
correct = 0

with torch.no_grad():

  for batch_features , batch_labels in test_loader:

    batch_features , batch_labels = batch_features.to(device) , batch_labels.to(device)

    outputs = vgg16(batch_features)
    _ , predicted = torch.max(outputs , 1)

    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()

print(correct/total)

0.9235833333333333


In [25]:
total = 0
correct = 0

with torch.no_grad():

  for batch_features , batch_labels in train_loader:

    batch_features , batch_labels = batch_features.to(device) , batch_labels.to(device)

    outputs = vgg16(batch_features)
    _ , predicted = torch.max(outputs , 1)

    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()

print(correct/total)

0.9950625
